# 03 — Modélisation, GridSearchCV & tracking MLflow

Ce notebook couvre **Milestone 1 & 4**.

- Setup MLflow
- Pipelines anti data leakage (preprocessing + SMOTE + modèle)
- GridSearchCV sur **score métier**
- Logging des résultats et artefacts (ROC, confusion matrix, modèle)

In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import sys, os
sys.path.insert(0, os.path.abspath(".."))  # monte d'un niveau (../)
from src.config import PATHS
from src.data import load_parquet
from src.metrics import optimal_threshold_cost, get_proba
from src.mlflow_utils import init_mlflow, track_run
from src.models import make_cv, gridsearch_dummy, gridsearch_lgbm_smote, gridsearch_xgb_smote, gridsearch_logreg_smote, gridsearch_rf_smote, run_gridsearch


## 1) Charger dataset préparé

In [2]:
df = load_parquet(PATHS.data_processed / "train_fe.parquet")

assert "TARGET" in df.columns, "Le dataset doit contenir TARGET (uniquement train)."
X = df.drop(columns=["TARGET"])
y = df["TARGET"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(set(X.dtypes))
print(X.select_dtypes(include='object').columns.tolist())



(79999, 125) (20000, 125)
{dtype('int64'), dtype('float64'), dtype('bool'), dtype('O')}
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


## 2) Init MLflow

In [3]:
init_mlflow()


C:\apps\anaconda3\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/26 10:34:34 INFO mlflow.tracking.fluent: Experiment with name 'home_credit' does not exist. Creating a new experiment.


In [4]:
cv = make_cv(2)

In [5]:
from src.models import build_preprocessor

print(X.shape, set(X.dtypes))

pre = build_preprocessor(X)
Xt = pre.fit_transform(X)
print(Xt.shape, Xt.dtype)
print(type(Xt))

(99999, 125) {dtype('int64'), dtype('float64'), dtype('bool'), dtype('O')}
(99999, 256) float64
<class 'numpy.ndarray'>


## 3) LightGBM

In [6]:
pipe, param_grid = gridsearch_lgbm_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_lightGBM = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_lightGBM, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_lightGBM = {
    "business_cost_test": threshold_info["cost"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_test, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
}


Fitting 2 folds for each of 8 candidates, totalling 16 fits
[LightGBM] [Info] Number of positive: 6474, number of negative: 73525
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.892046
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.471195
[LightGBM] [Debug] init for col-wise cost 0.014993 seconds, init for row-wise cost 0.035096 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019919 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Sparse Multi-Val Bin
[LightGBM] [Info] Total Bins 12203
[LightGBM] [Info] Number of data points in the train set: 79999, number of used features: 231
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080926 -> initscore=-2.429831
[LightGBM] [Info] Start training from score -2.429831
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [D

C:\apps\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Résultat

In [7]:

gs.best_params_, gs.best_score_, extra_metrics_lightGBM

({'model__colsample_bytree': 0.8,
  'model__learning_rate': 0.05,
  'model__max_depth': 8,
  'model__min_child_samples': 60,
  'model__n_estimators': 400,
  'model__num_leaves': 31,
  'model__subsample': 0.8},
 np.float64(-21760.0),
 {'business_cost_test': 10365.0,
  'business_score_cv': np.float64(-21760.0),
  'AUC_test': 0.758976383453669,
  'mean_cv_threshold': np.float64(0.07250000000000001)})

### MLFlow tracking

In [8]:
track_run(
    run_name="lightgbm",
    model=best_model_lightGBM,
    X_train=X_train, y_train=y_train,
    X_valid=X_test, y_valid=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_lightGBM,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
)

C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/26 10:36:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


C:\apps\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\apps\anaconda3\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:215: FutureWarning: The filesystem model registry backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri)
Successfully registered model 'LGBMClassifier'.
Created version '1' of model 'LGBMClassifier'.


## 4) Baseline DummyClassifier (référence)

In [9]:
pipe, param_grid = gridsearch_dummy(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_dummyClassif = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_dummyClassif, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

# CV results artifact (utile pour stabilité & soutenance)
cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_dummyClassif = {
    "business_cost_test": threshold_info["cost"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_test, y_proba),
}


Fitting 2 folds for each of 1 candidates, totalling 2 fits


### Résultat

In [10]:
gs.best_params_, gs.best_score_, extra_metrics_dummyClassif

({},
 np.float64(-32370.0),
 {'business_cost_test': 16190.0,
  'business_score_cv': np.float64(-32370.0),
  'AUC_test': 0.5})

### MLFlow tracking

In [11]:
track_run(
    run_name="dummy_baseline",
    model=best_model_dummyClassif,
    X_train=X_train, y_train=y_train,
    X_valid=X_test, y_valid=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_dummyClassif,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
)

C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/26 10:36:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully registered model 'DummyClassifier'.
Created version '1' of model 'DummyClassifier'.


## 5) XGBoost 

In [12]:
pipe, param_grid = gridsearch_xgb_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_xgBoost = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_xgBoost, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_xgBoost = {
    "business_cost_test": threshold_info["cost"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_test, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
}

Fitting 2 folds for each of 2 candidates, totalling 4 fits


### Résultat

In [13]:
gs.best_params_, gs.best_score_, extra_metrics_xgBoost

({'model__colsample_bytree': 0.8,
  'model__learning_rate': 0.05,
  'model__max_depth': 4,
  'model__min_child_weight': 5,
  'model__n_estimators': 500,
  'model__subsample': 0.8},
 np.float64(-21683.0),
 {'business_cost_test': 10421.0,
  'business_score_cv': np.float64(-21683.0),
  'AUC_test': 0.7574006331362592,
  'mean_cv_threshold': np.float64(0.4749999999999999)})

### MLFlow tracking

In [14]:
track_run(
    run_name="xgboost",
    model=best_model_xgBoost,
    X_train=X_train, y_train=y_train,
    X_valid=X_test, y_valid=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_xgBoost,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
)


C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/26 10:37:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully registered model 'XGBClassifier'.
Created version '1' of model 'XGBClassifier'.


### SAVE model to .joblib

In [15]:
import joblib

best_model = best_model_xgBoost #gs.best_estimator_

joblib.dump(best_model, "../model/model.joblib")

print("✅ Model saved to model.joblib")
print("Best params:", gs.best_params_)
print("Best score:", gs.best_score_)

✅ Model saved to model.joblib
Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__min_child_weight': 5, 'model__n_estimators': 500, 'model__subsample': 0.8}
Best score: -21683.0


In [25]:
# !pip freeze > requirements.txt

In [26]:
import sklearn
print(sklearn.__version__)

1.7.2


In [27]:
import imblearn
print(imblearn.__version__)

0.14.0


## 6) Logistic Regression + SMOTE

In [16]:
pipe, param_grid = gridsearch_logreg_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_logRegression = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_logRegression, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_logRegression = {
    "business_cost_test": threshold_info["cost"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_test, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
}

Fitting 2 folds for each of 2 candidates, totalling 4 fits


C:\apps\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Résultats

In [17]:
gs.best_params_, gs.best_score_, extra_metrics_logRegression

({'model__C': 1, 'model__solver': 'lbfgs'},
 np.float64(-27293.5),
 {'business_cost_test': 13658.0,
  'business_score_cv': np.float64(-27293.5),
  'AUC_test': 0.6343727992883056,
  'mean_cv_threshold': np.float64(0.095)})

### MLFlow tracking

In [18]:
track_run(
    run_name="logreg_smote",
    model=best_model_logRegression,
    X_train=X_train, y_train=y_train,
    X_valid=X_test, y_valid=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_logRegression,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
)

C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/26 10:40:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully registered model 'LogisticRegression'.
Created version '1' of model 'LogisticRegression'.


In [19]:
import mlflow

print("tracking uri =", mlflow.get_tracking_uri())

exp = mlflow.get_experiment_by_name("Default")
print("Default experiment =", exp)

runs = mlflow.search_runs()
print("Nb runs =", len(runs))
print(runs[["run_id", "experiment_id", "tags.mlflow.runName"]].tail(20))

tracking uri = file:./mlruns
Default experiment = None
Nb runs = 4
                             run_id       experiment_id tags.mlflow.runName
0  50a57743821941ba8ec026a762adda10  771618081702271174        logreg_smote
1  e118dd450f5e4dae9b05eb54b9ff60b4  771618081702271174             xgboost
2  113c17ca74c748048279114fa873c692  771618081702271174      dummy_baseline
3  19f5bb4946464d8ea496364cb19fe323  771618081702271174            lightgbm


## 7) RandomForest + SMOTE

In [20]:
pipe, param_grid = gridsearch_rf_smote(X_train, y_train, cv)
t0 = time.time()
gs = run_gridsearch(pipe, param_grid, X_train, y_train, cv)
fit_time = time.time() - t0

best_model_randomForest = gs.best_estimator_
t1 = time.time()
y_proba = get_proba(best_model_randomForest, X_test)
predict_time = time.time() - t1

threshold_info = optimal_threshold_cost(y_test.values, y_proba)
y_pred = (y_proba >= threshold_info["threshold"]).astype(int)

cv_results_df = pd.DataFrame(gs.cv_results_)

extra_metrics_randomForest = {
    "business_cost_test": threshold_info["cost"],
    "business_score_cv": gs.best_score_,
    "AUC_test": roc_auc_score(y_test, y_proba),
    "mean_cv_threshold": gs.cv_results_["mean_test_threshold"][gs.best_index_],
}

Fitting 2 folds for each of 1 candidates, totalling 2 fits


### Résultats

In [21]:
gs.best_params_, gs.best_score_, extra_metrics_randomForest

({'model__max_depth': 8,
  'model__max_features': 'sqrt',
  'model__min_samples_leaf': 20,
  'model__n_estimators': 200},
 np.float64(-22792.0),
 {'business_cost_test': 10928.0,
  'business_score_cv': np.float64(-22792.0),
  'AUC_test': 0.7348189894101715,
  'mean_cv_threshold': np.float64(0.0925)})

### MLFlow tracking

In [22]:
track_run(
    run_name="rf_smote",
    model=best_model_randomForest,
    X_train=X_train, y_train=y_train,
    X_valid=X_test, y_valid=y_test,
    params=gs.best_params_,
    extra_metrics=extra_metrics_randomForest,
    y_valid_proba=y_proba,
    y_valid_pred=y_pred,
    threshold_info=threshold_info,
    fit_time=fit_time,
    predict_time=predict_time,
    cv_results_df=cv_results_df,
)

C:\apps\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/26 10:41:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully registered model 'RandomForestClassifier'.
Created version '1' of model 'RandomForestClassifier'.


## 8) À faire

- Ajouter d'autres algos (GradientBoosting, XGBoost/LightGBM si autorisé)
- Faire un **fine-tuning** autour des meilleurs hyperparamètres
- Comparer via MLflow UI (runs)

# Dana Tests

## Étape 1 — Comparer les algorithmes (simple)
### 1 algorithme = 1 run MLflow

In [23]:
import mlflow
models = {
    "LogisticRegression": best_model_logRegression,
    "RandomForest": best_model_randomForest,
}

for name, model in models.items():
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

    with mlflow.start_run(run_name=name):
        mlflow.log_metric("auc", auc)
        mlflow.log_param("model", name)


C:\apps\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Étape 2 — Affiner le meilleur (toujours simple)
### Supposons que RandomForest gagne.
### 1 run = 1 réglage

In [24]:
from sklearn.ensemble import RandomForestClassifier
depths = [5, 10, 15]

for d in depths:
    rf = RandomForestClassifier(max_depth=d)
    rf.fit(X_train, y_train)
    auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

    with mlflow.start_run(run_name=f"RF_depth_{d}"):
        mlflow.log_metric("auc", auc)
        mlflow.log_param("max_depth", d)


ValueError: could not convert string to float: 'Cash loans'